Before you turn this problem in, make sure everything runs as expected. First, **restart the kernel** (in the menubar, select Kernel$\rightarrow$Restart) and then **run all cells** (in the menubar, select Cell$\rightarrow$Run All).

Make sure you fill in any place that says `YOUR CODE HERE` or "YOUR ANSWER HERE", as well as your name and collaborators below:

In [ ]:
NAME = ""
COLLABORATORS = ""

---

# Problem description

You have already classified satellite photos as *ship* / *no ship* twice: with Logistic Regression, and with **Dense** neural networks in which you varied the number of layers and their width.
In both, we first converted the photo to gray scale and **flattened** it into a vector of 6400 numbers.

In this assignment you will use **convolutional** neural networks (CNNs) on the same photos, keeping them as **images**: 80 x 80 pixels with 3 color channels, no flattening of the input and no conversion to gray scale.
You will run a set of experiments on the number of convolutional layers, the number of filters and the **filter size**, and compare what you find with your Dense results.

## Goal and learning objectives
- Understand why convolutional layers suit images, and why neither flattening nor gray-scale is needed
- Count the parameters of a CNN, and compute the **receptive field** of a stack of layers
- Run a controlled experiment grid (number of convolutional blocks, filters per layer, filter size) and compare it with your Dense experiments
- Use **early stopping**
- Practice disciplined model selection: choose on a validation set, touch the test set once

This is a checkpoint assignment: it verifies that you can build, train and compare CNNs and think about what you see. It is not meant to be a puzzle.

## Grading
The required parts are worth **200 points**. You can earn up to **100 additional points** for going above and beyond: the optional extensions at the end of the notebook and any further experiments of your own. In the required parts, written answers are graded on whether they are consistent with **your own results** and give specific support (a number from your table or a named configuration), not on length. Optional work is graded on whether it ran, on what you conclude from it, and on how carefully you handled the noise.

## How to run this notebook

**Toolkit:** any neural-network toolkit (Keras with any backend, TensorFlow, PyTorch, ...). Everything you are asked to *report* is toolkit-independent.

**Hardware: a GPU is recommended.** Convolutions are much more expensive than Dense layers.
- **A machine with a GPU, or Google Colab (free GPU runtime):** the 12 required networks took about 3 minutes of training (189 s) on a free Colab T4 when we ran the whole notebook there (about 5 to 6 minutes including starting up and loading the data). If it takes more than about 15 minutes, check with `helper.describe_hardware()` that the GPU is really attached.
- **A CPU-only machine:** the required grid takes roughly 10 to 40 minutes, depending on your CPU (7 to 11 minutes on a 16-core desktop). You may instead run the **reduced grid** (filter size 3 only; set `FULL_GRID = False` below). It is graded the same way; say so in your answers.
- Please be considerate with free Colab: the free tier **rations the GPU**. If you use too much GPU time in a day, Colab can deny you any GPU for the next 24 hours. So: edit the notebook and write your answers on a **CPU** runtime, switch to the **GPU** runtime only to run the grid, and **disconnect the runtime** (Runtime menu, "Disconnect and delete runtime") when the grid has finished. The grid code below saves each result to a file as it finishes, so a lost connection never forces you to start over.

**Running on Google Colab**
- Upload `helper.py` to **the same directory as your notebook** (the Colab Files panel, or the Drive folder holding the notebook).
- **Recommended: put `shipsnet.json` (344 MB) on your own Google Drive, once.** Upload it to `My Drive/Colab Notebooks/Data/` (do this while you are on a CPU runtime or from drive.google.com, before you switch to a GPU). Then mount the Drive in a cell (`from google.colab import drive; drive.mount('/content/drive')`; Google will ask for your permission). The helper looks in `Colab Notebooks/Data` and `Data` on your Drive automatically; if you used another folder, set `DATA_DIR` below to the folder that contains `shipsnet.json`.
- Uploading `shipsnet.json` to the Colab session through the *Files* panel also works in principle, but it is a 344 MB upload (slow, and it can fail on a poor connection) and the file disappears whenever the runtime is recycled, so you would have to upload it again. On a free GPU runtime that wastes rationed GPU time.
- Colab's Python and library versions change over time; use `helper.describe_hardware()` below to see what you have.

# Imports

In [ ]:
## Standard imports
import os
import math
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sklearn
from IPython.display import display

%matplotlib inline

In [ ]:
## Are we running on Colab?
try:
  from google.colab import drive
  IN_COLAB=True
except:
  IN_COLAB=False

if IN_COLAB:
  print("We're running Colab")

In [ ]:
## Where is the data?
# Leave as None if shipsnet.json is next to this notebook (or in ./Data).
# Otherwise set it to the folder that contains shipsnet.json, e.g. (Colab + Google Drive):
#   drive.mount('/content/drive')
#   DATA_DIR = "/content/drive/MyDrive/my_folder"
DATA_DIR = None

In [ ]:
## Load the helper module
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

# Optional convenience: pick up edits to helper.py without restarting the kernel.
# (Not available on every Python/IPython combination, e.g. Colab's, so failure is fine.)
try:
    get_ipython().run_line_magic("reload_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "1")
    AUTORELOAD = True
except Exception:
    AUTORELOAD = False

import helper as helper_module
if AUTORELOAD:
    get_ipython().run_line_magic("aimport", "helper")

helper = helper_module.Helper(data_dir=DATA_DIR)

In [ ]:
## Import your neural-network toolkit here (plus anything else you need)
# Tip (TensorFlow/Keras): the harmless C++ log messages can be silenced by setting
#   os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"   before importing keras/tensorflow
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
## What hardware do we have?
helper.describe_hardware()

# Set to False on a CPU-only machine to run the reduced grid (filter size 3 only)
FULL_GRID = True

# API for students

`helper.py` defines a class `Helper`; `helper = helper.Helper()` creates one.

- `getData`: get the labeled images: `data, labels = helper.getData()`
- `showData`: visualize labeled images: `helper.showData(data, labels)`
- `showFilters`: show convolution filters as small images (used in an *optional* part): `helper.showFilters(W)`
- `showWeights`: show weight vectors as images (optional, as in the Dense assignment)
- `describe_hardware`: report whether a GPU is visible
- `EarlyStopper`: early stopping for a training loop you write yourself (PyTorch); Keras has a built-in callback. See the recipe below.

# Get the data

In [ ]:
# Get the data
data, labels = helper.getData()
n_samples, width, height, channel = data.shape

print("Data shape: ", data.shape)
print("Labels shape: ", labels.shape)
print("Label values: ", np.unique(labels))

Your expected outputs should be the following

`Data shape: (4000, 80, 80, 3)`
`Labels shape: (4000,)`
`Label values: [0 1]`

(Loading the 344 MB file takes 20 to 30 seconds.)

In [ ]:
# Shuffle the data first
data, labels = sklearn.utils.shuffle(data, labels, random_state=42)

## Have a look at the data
This time we keep the three color channels.

In [ ]:
helper.showData(data[:25], labels[:25])

---
# Part 1: Before you run anything

Answer these **before** you run the experiments (write them in the cells below first; **Run All** is for your final check, after your answers are written), and do not change your answers afterwards. (Part of what we are testing is how good your intuition is.)

**Question 1 (written): why not flatten, and why not gray-scale?**
In the Dense assignments we flattened the images and converted them to gray scale. Here we do neither.
- What does flattening an image into a vector of 6400 numbers throw away, and why does a convolutional layer not need to flatten its input?
- Why is it fine (even useful) to keep the three color channels as input to a convolutional layer?

YOUR ANSWER HERE

**Question 2 (written): predict.**
Think about your best **Dense** network from the previous assignment. Predict, in a few sentences, how your best **CNN** will compare with it in (a) validation accuracy, (b) number of parameters and (c) training time, and why.

YOUR ANSWER HERE

---
# Prepare the data

## Split: train / validation / test
Exactly as in the Dense assignment, but the examples are now the *images* `data` of shape `(4000, 80, 80, 3)`.

**Question:** create `X_train, X_val, X_test, y_train, y_val, y_test` as follows:
- First split off 10% of the examples as the **test** set, with `random_state=42` (the same test set as in the earlier assignments)
- Then split the remaining 90% into train and validation so that the **validation** set has 400 examples (1/9 of what remains), with `random_state=42`

In [ ]:
from sklearn.model_selection import train_test_split

X, y = data, labels

# Create X_train, X_val, X_test, y_train, y_val, y_test
# YOUR CODE HERE
raise NotImplementedError()

print("X_train shape: ", X_train.shape)
print("X_val shape:   ", X_val.shape)
print("X_test shape:  ", X_test.shape)

# Self-check (given): sizes, and number of ships in each set
print("ships in train/val/test:", int(y_train.sum()), int(y_val.sum()), int(y_test.sum()))
assert (X_train.shape[0], X_val.shape[0], X_test.shape[0]) == (3200, 400, 400)
assert (int(y_train.sum()), int(y_val.sum()), int(y_test.sum())) == (812, 101, 87), "Right sizes but different examples: follow the order of the two splits described above"

Your expected outputs should be the following

`X_train shape:  (3200, 80, 80, 3)`
`X_val shape:    (400, 80, 80, 3)`
`X_test shape:   (400, 80, 80, 3)`

## Standardize, one mean and one standard deviation **per color channel**

**Question:** compute `mean` and `std` over the **training** images only, one value for each of the 3 color channels (so each has 3 values; shape `(1, 1, 1, 3)` is convenient). Create `X_train_s`, `X_val_s`, `X_test_s` as `float32` arrays equal to `(X - mean) / std`.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

print(X_train_s.shape, X_train_s.dtype, X_train_s.mean(axis=(0, 1, 2)).round(3), X_train_s.std(axis=(0, 1, 2)).round(3))

---
# Part 2: The architecture family, parameters, and receptive fields

## The networks you will build
A **block** is one or more convolutional layers, each `Conv2D(filters, k, padding="same")` with ReLU activation, followed by a **2 x 2 max pooling**. After the last block, **flatten** the feature maps and add a single sigmoid output unit.
(Flattening here comes at the *end*, after the convolutions have extracted features; it is the *input* we do not flatten.)

We describe a network by `blocks`, a list with one entry per block, each entry listing the filter sizes of that block's convolutional layers; every convolutional layer has the same number `filters` of filters. For example:
- `blocks = [[3], [3]]` : two blocks, each with a single 3 x 3 convolution
- `blocks = [[5], [5], [5]]` : three blocks, each with a single 5 x 5 convolution
- `blocks = [[3, 3], [3, 3]]` : two blocks, each with two 3 x 3 convolutions

In the experiment grid we use $L$ blocks that each contain one convolution of size $k$: `blocks = [[k]] * L`.

## Count the parameters by hand
A convolutional layer with `c_in` input channels, `filters` filters of size `k x k` has `(k * k * c_in + 1) * filters` parameters (the `+ 1` is the bias of each filter). Max pooling has no parameters.

The flattened feature maps feed the single output unit: that final Dense layer has `s * s * c + 1` parameters, where `s` is the spatial size after the last pooling and `c` the number of channels at that point. Inside a block the first convolution takes the block's input channels and every later convolution takes `filters` channels.

**Question:** write `count_cnn_params(blocks, filters, in_channels=3, size=80)`, returning the total number of parameters of the network described above (the input is `size x size x in_channels`; `padding="same"` keeps the spatial size, and each max pooling halves it). Compute it from the formulas; **do not build a model to get the answer**.

In [ ]:
def count_cnn_params(blocks, filters, in_channels=3, size=80):
    """Number of parameters of: [Conv(filters, k, same)+ReLU for k in block] + MaxPool(2) per block, Flatten, Dense(1)."""
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
# Given check with a multi-convolution block (if this fails, re-read the formulas above)
assert count_cnn_params([[3, 3]], 4) == 6661

## The receptive field
The **receptive field** of a unit is the size (in input pixels, along one side) of the region of the input that can influence it. One `k x k` convolution with stride 1 on the input itself has receptive field `k`. Stacking layers makes it larger, and a pooling layer (or any stride) makes later layers "see" more of the input per step.

Describe a stack of layers as a list of `(kernel_size, stride)` pairs in order (a 2 x 2 max pooling is `(2, 2)`).

**Question:** write `receptive_field(layer_specs)` returning the receptive field (in input pixels, along one side) of one unit in the output of the stack. Hint: track two numbers while walking through the layers, starting from `rf = 1` and `jump = 1` (`jump` is the distance in input pixels between neighboring units of the current layer). For each `(k, stride)`: first add `(k - 1) * jump` to `rf`, then multiply `jump` by `stride`.

In [ ]:
def receptive_field(layer_specs):
    """layer_specs: list of (kernel_size, stride); returns the receptive field along one side, in input pixels."""
    # YOUR CODE HERE
    raise NotImplementedError()

assert receptive_field([(5, 1)]) == 5
assert receptive_field([(3, 1), (2, 2), (3, 1)]) == 8      # conv 3, max-pool 2, conv 3

---
# Part 3: Train one network, with early stopping

## The training recipe (the same for *every* network in this notebook)

> **Read this table before you write any code, and check your code against it afterwards.** Every row is part of the experiment. The **random seed** and **batch shuffling** rows are the two that are easiest to get wrong, and they matter for the noise questions later.

| Setting | Value |
|---|---|
| Loss | binary cross-entropy |
| Optimizer | Adam, learning rate 0.001 |
| Batch size | 64 |
| Maximum number of epochs | 40 |
| **Early stopping** | monitor the **validation loss**; stop when it has not improved by at least `MIN_DELTA` = 0.001 for `PATIENCE` = 5 epochs in a row; **restore the weights of the best epoch** |
| Hidden-layer activation | ReLU |
| Output layer | 1 unit, sigmoid |
| **Random seed** | `SEED` (below); set it the way your toolkit does, e.g. `keras.utils.set_random_seed(SEED)` or `torch.manual_seed(SEED)` |
| Inputs | the standardized color images `X_train_s`, ... |
| **Batches** | **shuffle the training examples every epoch** (reshuffle, not one fixed order); keep the last, smaller batch |
| Reported accuracy | evaluate the **final model (after restoring the best weights)** on the whole set |
| Initialization, other optimizer settings | your toolkit's defaults |

**Before you rely on `SEED`:** work out what in *your* toolkit's training code is random, and check that your seed setting really controls all of it. Optional C asks you to write this down.

**Early stopping.** Keras: `keras.callbacks.EarlyStopping(monitor="val_loss", min_delta=MIN_DELTA, patience=PATIENCE, restore_best_weights=True)`, passed to `fit` together with `validation_data=(X_val_s, y_val)`.
PyTorch (and other hand-written loops): `helper.EarlyStopper(patience=PATIENCE, min_delta=MIN_DELTA)`; read its docstring (`help(helper.EarlyStopper)`).

Note that the validation set is now used twice: to decide when to stop and (later) to choose among configurations. Only the **test** set stays untouched until the very end.

In [ ]:
SEED = 0
MAX_EPOCHS = 40
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
MIN_DELTA = 1e-3
PATIENCE = 5

**Question:** write `build_cnn(blocks, filters)`, returning an untrained (and, if your toolkit needs it, compiled) network of the architecture family of Part 2 with ReLU activations and a single sigmoid output.
Then write `run_experiment(blocks, filters, seed=SEED)` that builds the network, trains it with the recipe (including early stopping) on `X_train_s, y_train`, and returns a dictionary with the keys
- `"model"`, the trained model (best weights restored)
- `"num_params"`, the number of parameters **as reported by your toolkit**
- `"train_acc"`, `"val_acc"`, the final accuracy on the training and validation examples
- `"epochs_run"`, the number of epochs actually trained (including the epochs after the best one)
- `"secs"`, the wall-clock training time in seconds

In [ ]:
def build_cnn(blocks, filters):
    # YOUR CODE HERE
    raise NotImplementedError()

def run_experiment(blocks, filters, seed=SEED):
    # YOUR CODE HERE
    raise NotImplementedError()

# Sanity check (this cell takes about a minute): your count_cnn_params must agree with your toolkit
info = run_experiment([[3], [3]], 8)
print("params: formula {f}, toolkit {t}; epochs run {e}; train acc {a:.3f}; val acc {v:.3f}; {s:.0f}s".format(
    f=count_cnn_params([[3], [3]], 8), t=info["num_params"], e=info["epochs_run"],
    a=info["train_acc"], v=info["val_acc"], s=info["secs"]))
assert info["num_params"] == count_cnn_params([[3], [3]], 8)

---
# Part 4: The experiment grid

Run **every combination** of
- $L \in \{1, 2, 3\}$ blocks, each containing a single convolution (`blocks = [[k]] * L`)
- `filters` $\in \{8, 16\}$ filters in every convolutional layer
- filter size $k \in \{3, 5\}$ (only $k = 3$ if you set `FULL_GRID = False`)

That is 12 networks (6 for the reduced grid). For each, record the number of parameters, the training and validation accuracy, the number of epochs run, and the time.

**Question:** create a `pandas` DataFrame named `results` with one row per experiment and the columns `L`, `filters`, `k`, `num_params`, `train_acc`, `val_acc`, `epochs_run`, `secs`.
- Before you run each experiment compute `count_cnn_params` and **assert** that it equals the number your toolkit reports.
- **Save the table to `cnn_results.csv` after every row, and skip rows already in that file when the cell is rerun** (so that a lost connection or an interrupted run costs you nothing). Delete the file if you want to start from scratch. (To test it, interrupt the cell with the stop button and run it again.)
- Don't tune anything: the recipe is fixed so that only the architecture changes.
- **Noise:** training involves randomness and the validation set has only 400 examples. Differences of about 1.5 percentage points or less are within noise. Expect an outlier or two; if you see one, say so instead of building a story around it.

In [ ]:
CSV_PATH = "cnn_results.csv"
rows = pd.read_csv(CSV_PATH).to_dict("records") if os.path.exists(CSV_PATH) else []
done = {(r["L"], r["filters"], r["k"]) for r in rows}
kernel_sizes = (3, 5) if FULL_GRID else (3,)

# YOUR CODE HERE
raise NotImplementedError()

results = pd.DataFrame(rows)
results

## Your Dense results
Enter the numbers from **your own** best Dense network in the previous assignment (the configuration you chose there), so that we can compare. If you did not do that assignment, use your best guess and say so.

In [ ]:
DENSE_BEST_PARAMS = None       # number of parameters of your best Dense network
DENSE_BEST_VAL_ACC = None      # its validation accuracy
DENSE_BEST_TEST_ACC = None     # its test accuracy
DENSE_BEST_SECS = None         # its training time in seconds, if you recorded it

## Look at the results
The next cell (already written) plots your `results` (it needs `results` with the columns above).

In [ ]:
results["gap"] = results["train_acc"] - results["val_acc"]
display(results.round(3))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for (k, filters), grp in results.groupby(["k", "filters"]):
    axes[0].plot(grp.L, grp.val_acc, marker="o", label="k={k}, {f} filters".format(k=k, f=filters))
axes[0].set_xlabel("number of blocks L"); axes[0].set_ylabel("validation accuracy"); axes[0].set_xticks([1, 2, 3]); axes[0].legend(fontsize=7)
axes[1].scatter(results.num_params, results.val_acc, label="CNN")
if DENSE_BEST_PARAMS is not None and DENSE_BEST_VAL_ACC is not None:
    axes[1].scatter([DENSE_BEST_PARAMS], [DENSE_BEST_VAL_ACC], marker="*", s=200, color="red", label="your best Dense")
axes[1].set_xscale("log"); axes[1].set_xlabel("number of parameters"); axes[1].set_ylabel("validation accuracy"); axes[1].legend()
axes[2].scatter(results.secs, results.val_acc)
axes[2].set_xlabel("training time (s)"); axes[2].set_ylabel("validation accuracy")
fig.tight_layout();

**Questions (written).** Use your table and plots, and remember the noise warning. A short paragraph per question is plenty; refer to specific rows of your table.

1. **Depth.** What happens to validation accuracy as the number of blocks $L$ increases?
2. **Filters and filter size.** What do the number of filters and the filter size change (accuracy, parameters, time)? Which of the three dimensions ($L$, filters, $k$) matters most?
3. **Parameters.** Split the parameters of your $L=1$ and $L=3$ networks (same filters and $k$) into the convolutional part and the final Dense layer. Which part dominates, and how does the total change with $L$ and why? Compare with the Dense case, where the first layer dominated.
4. **In-sample versus out-of-sample.** How do training and validation accuracy compare? Compare the numbers of parameters with the 3200 training examples.
5. **Versus your Dense network.** Compare your best CNN with your best Dense network (accuracy, parameters and, if you recorded it, training time). Was your prediction in Part 1 right? What surprised you?
6. **Early stopping.** How many epochs did the networks run? What would you lose, and what would you gain, by training all 40 epochs without early stopping? (You may run one network for all 40 epochs to check.)
7. **Pooling.** Every block ends with a 2 x 2 max pooling. Give at least two consequences of that choice that you can see in your own table or by using your `receptive_field` function. If a block had no pooling, what would change?

YOUR ANSWER HERE

---
# Part 5: Receptive fields: two 3 x 3 layers versus one 5 x 5

**Questions (code):** consider two networks with two blocks and 16 filters each:
- **A:** `blocks = [[5], [5]]` (one 5 x 5 convolution per block)
- **B:** `blocks = [[3, 3], [3, 3]]` (two 3 x 3 convolutions per block)

Compute, with your functions: `rf_A` and `rf_B`, the receptive field of the **convolutions of one block** (without the pooling); and `params_A`, `params_B`, the number of parameters of the two networks. Then run both with `run_experiment` and store the outcomes in a DataFrame `rf_results` with one row per network and the columns `name`, `num_params`, `val_acc`, `epochs_run`, `secs`.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

print("receptive field of one block's convolutions: A =", rf_A, ", B =", rf_B)
print("parameters: A =", params_A, ", B =", params_B)
rf_results

**Questions (written):**
1. What do you find for the receptive fields of A and B? Explain in a sentence *why* (what does the second 3 x 3 layer "see"?).
2. Compare the parameters of the convolutions in A and B for `c` filters and `c` input channels (that is, every convolution in A and B has `c` input channels and `c` filters): which has fewer, and by what factor in general (for one 5 x 5 versus two 3 x 3 layers)?
3. B also has an extra ReLU. Does B perform differently from A in your table? Considering the noise, what would you conclude, and what would you need to be more sure?
4. Extrapolate: what stack of 3 x 3 layers has the receptive field of one 7 x 7 layer, and how do the parameter counts compare?

YOUR ANSWER HERE

---
# Part 6: Choose a model, then use the test set once

**Question:**
- Choose the configuration (`best_L`, `best_filters`, `best_k`) among the networks in `results` as follows. Count the correct validation predictions of each: `n_correct = round(val_acc * 400)`. Let `top` be the largest `n_correct`. Every configuration with `n_correct >= top - 6` is a candidate (6 examples is 1.5 points of 400); of these, choose the one with the **fewest parameters**. (In this family more blocks mean fewer parameters, so the rule tends to favor deeper networks.)
- Train that configuration with `run_experiment` and evaluate it on the **test** set: set `test_acc_cnn`.

This is the **only** time in this notebook you may evaluate on the test set. Because of noise, your chosen configuration may differ from a classmate's; what matters is that it follows the rule when applied to **your** table.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

print("Chosen network: L={L}, filters={f}, k={k}".format(L=best_L, f=best_filters, k=best_k))
print("Test accuracy of the chosen CNN: {a:.3f}".format(a=test_acc_cnn))
if DENSE_BEST_TEST_ACC is not None:
    print("Test accuracy of your best Dense network: {a:.3f}".format(a=DENSE_BEST_TEST_ACC))

**Questions (written):**
1. Is the test accuracy of your chosen CNN similar to its validation accuracy? Would you expect it to be a little lower, and why?
2. Using what you have seen, say in your own words **why** the CNN beats the Dense network on this problem. State a hypothesis and what evidence you could collect to support or refute it. (You do not have to collect it; Optional A below shows one way.)

YOUR ANSWER HERE

---
# Optional extensions

These are optional, but they are how you earn the additional 100 points. They also build intuition.

## Optional A: Look at what the filters learned
Take the best network from Part 6 (or train one with `run_experiment`). Extract the weights of its **first** convolutional layer (for a `k x k` filter on 3 color channels: `k x k x 3` per filter), and show them with `helper.showFilters(W)` (Keras layout `(k, k, 3, filters)`; PyTorch stores `(filters, 3, k, k)`, so pass `W.transpose(2, 3, 1, 0)` of the numpy array).

**Tip:** weights start as random numbers and a small network trained for a few epochs may not move far from them. Also look at the filters **before training** (build the network with the same seed and do not train it, or keep a copy of the initial weights) and at the **change** `W_trained - W_initial`. Each filter is rescaled independently, so noise can look dramatic.

**Written:** what do you see (edge detectors? color detectors? blobs? just noise?), and how does it compare with the first-layer weight images of your Dense network (one weight per pixel position)? Describe what *your own* picture shows. Why can you show *these* filters but not the weights of later layers as pixel images?

In [ ]:
# YOUR CODE HERE (optional)

YOUR ANSWER HERE

## Optional B: Do we need color?
Convert the images to gray scale (as in the earlier assignments, keeping one channel) and run the same architecture on them (one input channel). Compare with the color version using **several seeds** (at least 3) and report means and spreads.

**Written:** does gray scale lose accuracy? Is the difference larger than the noise? Why might color matter (or not) for ships?

In [ ]:
# YOUR CODE HERE (optional)

## Optional C: Measure the noise
Repeat the whole grid (or a clearly stated subset) with **three different seeds**. Report mean and (sample) standard deviation of the validation accuracy per configuration. Revisit at least one of your earlier conclusions with the spread in hand.

**Written:** (1) List every place in *your* training code where randomness enters. For each, say whether your seed setting controls it, and how you checked (for example, same seed twice, different seeds). (2) In an experiment with "several seeds", should the train/validation split also change from seed to seed? Say what each choice would measure.

In [ ]:
# YOUR CODE HERE (optional)

YOUR ANSWER HERE

## Optional D: Bigger receptive fields and global pooling
(1) Compare `[[7], [7]]` with `[[3, 3, 3], [3, 3, 3]]` (same receptive field per block, 16 filters): parameters, accuracy, time (several seeds). (2) Replace the final `Flatten` with a **global average pooling** (average each feature map into one number) before the output unit. How does that change the number of parameters and the accuracy? What does global average pooling assume about *where* in the image the ship is? (Recall that the ships in this data set are centered.)

In [ ]:
# YOUR CODE HERE (optional)

## Optional E: Your own experiments
Anything you are curious about. Some ideas: flip or rotate the training images as extra training examples (data augmentation); add dropout or batch normalization; try another early-stopping patience; look at the images your best network gets wrong (error analysis); try stride-2 convolutions instead of max pooling; test whether your CNN really needs the pixel layout (apply **one fixed random permutation** to the pixel positions of every image, in train, validation and test, and retrain), or whether it tolerates ships at other positions (evaluate on images shifted by a few pixels).

For each experiment state a **hypothesis**, change **one thing**, compare with a baseline **while taking the noise into account**, and write a short paragraph on what you found.

In [ ]:
# YOUR CODE HERE (optional)

# Done

In [ ]:
print("Done")